In [ ]:
!pip install requests azure-storage-blob tqdm
import os
import json
import requests
from tqdm import tqdm
from azure.storage.blob import BlobServiceClient, ContentSettings

# -----------------------
# CONFIG
# -----------------------
# Two Google API keys
street_view_keys = [
    os.environ.get("STREET_VIEW_API_KEY1", "YOUR_STREET_VIEW_API_KEY1"),
    os.environ.get("STREET_VIEW_API_KEY2", "YOUR_STREET_VIEW_API_KEY2"),
]
azure_api_key = os.environ.get("AZURE_STORAGE_CONN_STR", "")


blob_path = "london"
prefix = "training"
container_name = blob_path

remote_meta_folder = "RandomSamplingMetadata"
target_folder = "training/Images"
angles = [0, 90, 180, 270]
image_size = "640x640"
request_timeout = 20

checkpoint_file = "image_download_checkpoint.json"

# -----------------------
# Azure setup
# -----------------------
blob_service = BlobServiceClient.from_connection_string(azure_api_key)
container_client = blob_service.get_container_client(container_name)

def upload_bytes_to_blob(data: bytes, blob_rel_path: str, content_type="image/jpeg"):
    blob_client = container_client.get_blob_client(blob_rel_path)
    blob_client.upload_blob(data, overwrite=False,  # skip if already exists
                            content_settings=ContentSettings(content_type=content_type))

# -----------------------
# Google Street View fetch
# -----------------------
def fetch_streetview_image(pano_id, heading, key, pitch=0):
    base_url = "https://maps.googleapis.com/maps/api/streetview"
    params = {
        "size": image_size,
        "pano": pano_id,
        "heading": heading,
        "pitch": pitch,
        "key": key,
    }
    r = requests.get(base_url, params=params, timeout=request_timeout)
    if r.status_code == 200:
        return r.content
    return None

# -----------------------
# Checkpoint helpers
# -----------------------
def load_checkpoint():
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as fh:
            return json.load(fh)
    return {"done": set()}

def save_checkpoint(cp):
    with open(checkpoint_file, "w", encoding="utf-8") as fh:
        json.dump({"done": list(cp["done"])}, fh, indent=2)

# -----------------------
# Main
# -----------------------
def main():
    # List metadata blobs
    blobs = container_client.list_blobs(name_starts_with=f"{prefix}/{remote_meta_folder}")
    meta_files = sorted([b.name for b in blobs if b.name.endswith(".json")])

    total_files = len(meta_files)
    print(f"Found {total_files} metadata JSON files.")

    if total_files < 5000:
        print("Warning: fewer than 5000 metadata files available!")

    # Split into two halves
    first_half = meta_files[:2500]
    second_half = meta_files[2500:5000]

    # Load checkpoint
    checkpoint = load_checkpoint()
    done = set(checkpoint["done"])

    def process_files(files, api_key, key_label):
        count_uploaded = 0
        for meta_blob_name in tqdm(files, desc=f"Downloading images with {key_label}"):
            if meta_blob_name in done:
                continue

            blob_client = container_client.get_blob_client(meta_blob_name)
            meta_json = json.loads(blob_client.download_blob().readall())
            pano_id = meta_json.get("metadata_response", {}).get("pano_id")
            if not pano_id:
                continue

            seq = meta_json.get("seq", "na")

            for heading in angles:
                out_name = f"{prefix}/{target_folder}/{seq}_{heading}.jpg"
                blob_client_out = container_client.get_blob_client(out_name)

                if blob_client_out.exists():  # skip if already uploaded
                    continue

                img_bytes = fetch_streetview_image(pano_id, heading, api_key)
                if not img_bytes:
                    continue
                upload_bytes_to_blob(img_bytes, out_name)
                count_uploaded += 1

            # mark metadata file as completed
            done.add(meta_blob_name)
            save_checkpoint({"done": list(done)})

        return count_uploaded

    # Run both halves
    count_key1 = process_files(first_half, street_view_keys[0], "API Key 1")
    count_key2 = process_files(second_half, street_view_keys[1], "API Key 2")

    print(f"Finished: {count_key1} new images with API Key 1, {count_key2} new images with API Key 2")

if __name__ == "__main__":
    main()